In [1]:
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

In [2]:
%cd ../scripts/
import sys
import os
sys.path.append("../")
sys.path.append("../Modules")
import analysis

In [3]:
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-17-09-57-all3_stas_multiple_seeds_NoMapping/"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-19-11-31-stas_final_hopefully"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-19-21-16-stas_mapping_less_rhyth"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-20-10-53-stas_final_hopefully"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-20-13-37-checking_lower_rhyth"
# simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-20-13-56-checking_lower_rhyth"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-20-20-34-full_sim_rhyth_across_seeds"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-08-33-reduced_sim_tuning"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-12-07-tuning_detailed_rhyth_fr"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-13-50-tuning_detailed_rhyth_fr"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-14-07-tuning_detailed_rhyth_fr"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-14-49-tuning_detailed_rhyth_fr"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-15-34-tuning_detailed_rhyth_fr"
simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-16-04-tuning_detailed_rhyth_fr"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-16-42-tuning_detailed_rhyth_fr"

# simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-21-18-59-reduced_rhyth_fr"

simulations_dir = '/home/drfrbc/Neural-Modeling/scripts/2025-03-23-08-10-Basal_density_1035'

# simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-24-10-24-complex_baseline_fr_50sec"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-24-13-31-complex_rhyth_50sec"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-25-20-01-testing_decrease_synapse_densities_apical"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-26-11-42-tuning_tuft_syn_densities_while_decreasing"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-26-11-59-tuning_tuft_syn_densities_while_decreasing"

simulations_dir = "/home/drfrbc/Neural-Modeling/scripts/2025-03-26-12-15-0_tuft_syn_densities_while_decreasing"


In [4]:
sim_index = input(f"Specify the simulation index (0-{len(os.listdir(simulations_dir))-1}): from among simulations {os.listdir(simulations_dir)}")
sim_directory = os.path.join(simulations_dir, os.listdir(simulations_dir)[int(sim_index)])
print(f"Loading data from {sim_directory}")

def load_sim(sim_directory):
    # load parameters
    parameters = analysis.DataReader.load_parameters(sim_directory)

    # load recorded data
    sim_data = {
        'v': analysis.DataReader.read_data(sim_directory, "v").T,
        # 'hva': analysis.DataReader.read_data(sim_directory, "ica_Ca_HVA").T,
        # 'lva': analysis.DataReader.read_data(sim_directory, "ica_Ca_LVAst").T,
        # 'ih': analysis.DataReader.read_data(sim_directory, "ihcn_Ih").T,
        # 'nmda': analysis.DataReader.read_data(sim_directory, "i_NMDA").T,
        # 'na': analysis.DataReader.read_data(sim_directory, "gNaTa_t_NaTa_t").T,
        'spktimes': analysis.DataReader.read_data(sim_directory, "soma_spikes")[0][:],
        'spkinds': np.sort((analysis.DataReader.read_data(sim_directory, "soma_spikes")[0][:] * 10).astype(int)),
        # 'na_df': pd.read_csv(os.path.join(sim_directory, 'na.csv')),
        # 'ca_df': pd.read_csv(os.path.join(sim_directory, 'ca.csv')),
        # 'nmda_df': pd.read_csv(os.path.join(sim_directory, 'nmda.csv'))
    }

    # load segment information
    seg_data = pd.read_csv(os.path.join(sim_directory, "segment_data.csv"))
    seg_data['Sec ID'] = seg_data['idx_in_section_type']
    seg_data['Type'] = seg_data['section']
    seg_data['Coord X'] = seg_data['pc_0']
    seg_data['Coord Y'] = seg_data['pc_1']
    seg_data['Coord Z'] = seg_data['pc_2']
    elec_dist = pd.read_csv(os.path.join(sim_directory, f"elec_distance_{'soma'}.csv"))
    seg_data['Elec_distance'] = elec_dist['25_active']
    elec_dist = pd.read_csv(os.path.join(sim_directory, f"elec_distance_{'nexus'}.csv"))
    seg_data['Elec_distance_nexus'] = elec_dist['25_active']
    Xs = []
    for seg in seg_data['seg']:
        Xs.append(seg.split('(')[-1].split(')')[0])
    seg_data['X'] = Xs

    # continue
    seg_data['segmentID'] = seg_data.index

    seg_data['Sec ID'] = seg_data['Sec ID'].astype(int)
    seg_data['X'] = seg_data['X'].astype(float)
    seg_data['Elec_distanceQ'] = 'None'

    seg_data.loc[seg_data.Type=='dend','Elec_distanceQ'] = pd.qcut(seg_data.loc[seg_data.Type=='dend','Elec_distance'], 10, labels=False)
    seg_data.loc[seg_data.Type=='apic','Elec_distanceQ'] = pd.qcut(seg_data.loc[seg_data.Type=='apic','Elec_distance'], 10, labels=False)
    return parameters, sim_data, seg_data, elec_dist

parameters, sim_data, seg_data, elec_dist = load_sim(sim_directory)

In [5]:
# plot soma voltage
time_points=np.arange(0,int(parameters.h_tstop/parameters.h_dt))
time_points = time_points[0:49999]
# time_points = time_points[100000:149999]
# time_points = time_points[450000:499999]
colors = ['r*', 'g*', 'b*', 'm*', 'y*', 'k*']
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 2)
plt.plot(sim_data['v'][time_points, 0], colors[-1].split('*')[0])
plt.ylim([-90, 10])
plt.title(f'SOMA Voltage at index {0}')
plt.xlabel(f'Timesteps ({parameters.h_dt} ms)')

plt.show()

In [6]:
# spike rate

# Function to count the first occurrence of True in consecutive trues
def count_first_true(arr):
    return np.sum((arr > -10) & np.concatenate(([True], arr[:-1] <= -10)))

# Count the number of spikes in the soma
soma_spike_count = count_first_true(sim_data['v'][:, 0])
print(f"Soma spike count {soma_spike_count}")
sim_duration = len(sim_data['v'][:, 0])
print(f"Simulation duration in timesteps: {sim_duration}")
sim_duration_in_sec = sim_duration / ((1/parameters.h_dt) * 1000)
print(f"Simulation duration in seconds: {sim_duration_in_sec}")
soma_spike_rate = soma_spike_count / (len(sim_data['v'][:, 0]) / 10000)
print(f"Soma firing rate {round(soma_spike_rate, 3)}")

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_segments(seg_data, special_indices, special_colors, title_suffix="", save_file = None, show=False):
    # Calculate the axis limits
    all_coords_x = seg_data['Coord X'].tolist()
    all_coords_y = seg_data['Coord Y'].tolist()
    x_min, x_max = min(all_coords_x), max(all_coords_x)
    y_min, y_max = min(all_coords_y), max(all_coords_y)

    for i, segs in enumerate([seg_data]):
        plt.figure()
        plt.scatter(segs['Coord X'], segs['Coord Y'], s=0.1)
        for j, ind in enumerate(special_indices):
            plt.plot(segs.loc[segs.segmentID.isin([ind]), 'Coord X'], 
                     segs.loc[segs.segmentID.isin([ind]), 'Coord Y'], special_colors[j])
        
        plt.title(f"Segments {title_suffix}" if i == 0 else f"Segments {title_suffix}")
        plt.xlim(x_min, x_max)
        plt.ylim(y_min, y_max)
        if save_file:
            plt.savefig(f"{save_file}")
        if show:
            plt.show()

def plot_voltage(sim_data, indices, colors, title_suffix="", time_points=time_points, save_file = None, show=False):
    colors = [color.split('*')[0] for color in colors]
    for i, idx in enumerate(indices):
        plt.figure(figsize=(12, 6))

        # plt.subplot(1, 1, 1)
        plt.plot(sim_data['v'][time_points, idx], colors[i])
        plt.ylim([-90, 10])
        plt.title(f'Voltage at index {idx} {title_suffix}')
        
        # plt.subplot(1, 2, 2)
        # plt.plot(sim_data['v'][time_points, segment_mapping[idx]], colors[i])
        # plt.ylim([-90, 10])
        # plt.title(f'Refactored Model - Voltage at index {segment_mapping[idx]} {title_suffix}')
        if save_file:
            plt.savefig(f"{save_file}_{idx}")
        if show:
            plt.show()


# Filter for specific segment types
apic_segs= seg_data[seg_data['Type']=='apic']

# Plot segments
# apic_indices = [1500, 1400, 1900, 2000, 2500, 1800]
# apic_indices = np.random.choice(apic_segs.segmentID, 6)
apic_indices = [1402, 1203, 1842, 1533, 2601, 1887]
# colors = ['r*', 'g*', 'b*', 'm*', 'y*', 'k*']
# apic_indices = np.arange(1200, 2550, 50)
# apic_indices = [2400]
colors = ['r*', 'g*', 'b*', 'm*', 'y*', 'k*']* 20
plot_segments(apic_segs, apic_indices, colors, title_suffix="(Apical)", save_file=os.path.join(sim_directory, 'apic_segs'), show=True)

# Plot voltage
plot_voltage(sim_data, apic_indices, colors, title_suffix="(Apical)", save_file=os.path.join(sim_directory, 'apical_voltages'), show=True)

# Now for dendritic segments
dend_segs = seg_data[seg_data['Type']=='dend']

# Plot segments
# dend_indices = [10, 20, 30, 40, 50, 60]
# dend_indices = np.arange(10, 1000, 50)
# dend_indices = np.random.choice(dend_segs.segmentID, 6)
dend_indices = [689, 220, 635, 186, 777, 720]
plot_segments(dend_segs, dend_indices, colors, title_suffix="(Dendritic)", save_file=os.path.join(sim_directory, 'basal_segs'), show=True)

# Plot voltage
plot_voltage(sim_data, dend_indices, colors, title_suffix="(Dendritic)", save_file=os.path.join(sim_directory, 'basal_voltages'), show=True)

In [ ]:
# # generate figures for all simulations

# for sim_dir in os.listdir(simulations_dir):
#     sim_directory = os.path.join(simulations_dir, sim_dir)
#     if not os.path.exists(os.path.join(sim_directory, 'parameters.pickle')):
#         continue

#     parameters, sim_data, seg_data, elec_dist = load_sim(sim_directory)
#     plot_segments(apic_segs, apic_indices, colors, title_suffix="(Apical)", save_file=os.path.join(sim_directory, 'apic_segs'), show=False)

#     # Plot voltage
#     plot_voltage(sim_data, apic_indices, colors, title_suffix="(Apical)", save_file=os.path.join(sim_directory, 'apical_voltages'), show=False)

#     # Now for dendritic segments
#     dend_segs = seg_data[seg_data['Type']=='dend']

#     # Plot segments
#     # dend_indices = [10, 20, 30, 40, 50, 60]
#     # dend_indices = np.arange(10, 1000, 50)
#     # dend_indices = np.random.choice(dend_segs.segmentID, 6)
#     # dend_indices = [689, 220, 635, 186, 777, 720]
#     plot_segments(dend_segs, dend_indices, colors, title_suffix="(Dendritic)", save_file=os.path.join(sim_directory, 'basal_segs'), show=False)

#     # Plot voltage
#     plot_voltage(sim_data, dend_indices, colors, title_suffix="(Dendritic)", save_file=os.path.join(sim_directory, 'basal_voltages'), show=False)

In [ ]:
np.random.choice(apic_segs.segmentID, 10)

In [ ]:
apic_segs

In [ ]:
np.round(np.arange(len(apic_segs.segmentID)/10, len(apic_segs.segmentID), len(apic_segs.segmentID)/10))

In [ ]:
# list(apic_segs.segmentID)[np.round(np.arange(len(apic_segs.segmentID)/10, len(apic_segs.segmentID), len(apic_segs.segmentID)/10))]

In [ ]:
parameters.synapse_mapping

In [ ]:
parameters.rhyth_depth_inh_distal

Calculate FFT of voltage to look for artifcats from rhythmic inhibition

In [ ]:
# check PSD

from scipy.signal import welch
signal = sim_data['v'][:,0] #1644

# Compute PSD using Welch's method
f, Pxx = welch(signal, fs=10000, nperseg=10000)

Pxx = Pxx[f<100]
f=f[f<100]

# Plot the result
# plt.subplots(1,2)
# plt.subplot(121)
# plt.plot(signal)

# plt.subplot(122)
plt.semilogy(f, Pxx)
plt.scatter(f[f-16==min(abs(f-16))], Pxx[f-16==min(abs(f-16))], c='r', label=f'{f[f-16==min(abs(f-64))]} Hz')
plt.scatter(f[abs(f-64)==min(abs(f-64))], Pxx[abs(f-64)==min(abs(f-64))], c='orange', label=f'{f[abs(f-64)==min(abs(f-64))]} Hz')
# plt.semilogy(f, 100/f)
plt.title('Power Spectral Density using Welch\'s Method')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [V**2/Hz]')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
f[f-16==min(abs(f-16))]

In [ ]:
f

In [ ]:

f[Pxx == max(Pxx)]

Spike raster

In [ ]:
# read synapse_data.h5 from our simulations.
def read_synapse_distribution_file(sim_directory):
    """
    Reads the synapse_data.h5 file and loads its datasets into a Pandas DataFrame.

    Parameters:
        sim_directory (str): Path to the directory containing synapse_data.h5.

    Returns:
        pd.DataFrame: A DataFrame where each column corresponds to a dataset in the HDF5 file.
    """
    # Construct the full file path
    synapse_file_path = os.path.join(sim_directory, 'synapse_data.h5')

    # Check if the file exists
    if not os.path.exists(synapse_file_path):
        raise FileNotFoundError(f"File not found: {synapse_file_path}")

    # Dictionary to temporarily store the data
    synapse_data = {}

    # Read the HDF5 file
    with h5py.File(synapse_file_path, 'r') as h5f:
        # Load all datasets into the dictionary
        for key in h5f.keys():
            synapse_data[key] = h5f[key][()]  # Load dataset into memory as NumPy array

    # Convert the dictionary into a Pandas DataFrame
    synapse_df = pd.DataFrame(synapse_data)

    return synapse_df

def read_transfer_impedance_file(sim_directory, loc='soma'):
    # loc can be nexus
    imp_file = os.path.join(sim_directory, f"elec_distance_{loc}.csv")
    impedance_data = pd.read_csv(imp_file)
    return impedance_data

def add_seg_info_to_syn_data(syn_data, seg_data):
    """
    Adds segment information (Distance and section) from seg_data to syn_data
    using the seg_id as a lookup.

    Parameters:
        syn_data (pd.DataFrame): DataFrame containing synapse data with 'seg_id'.
        seg_data (pd.DataFrame): DataFrame containing segment data.

    Returns:
        pd.DataFrame: Updated syn_data with 'Distance' and 'section' columns added.
    """
    # Set 'Unnamed: 0' as the index in seg_data for easy lookup
    seg_data_indexed = seg_data.set_index('Unnamed: 0')

    # Use .loc to map the segment information to syn_data based on seg_id
    syn_data['Distance'] = syn_data['seg_id'].map(seg_data_indexed['Distance'])
    syn_data['section'] = syn_data['seg_id'].map(seg_data_indexed['section'])
    syn_data['soma_trans_imp'] = syn_data['seg_id'].map(seg_data_indexed['soma_trans_imp'])
    syn_data['seg_L'] = syn_data['seg_id'].map(seg_data_indexed['L'])

    # Convert the 'synapse_type' column from bytes to strings

# read synapses from our simulation and ben's
syn_data = read_synapse_distribution_file(sim_directory)

# read segment data from our simulation
seg_data = pd.read_csv(os.path.join(sim_directory, "segment_data.csv"))

# read transfer impedances
imp_df = read_transfer_impedance_file(sim_directory, loc='soma')
seg_data['soma_trans_imp'] = imp_df.beta_active

# add segment information of synapse location
# syn_data = add_seg_info_to_syn_data(syn_data, seg_data)

In [ ]:
print(syn_data)

In [ ]:
syn_data